# 17 — Sarkazm jako 9. etykieta (TwitterEmo)

Sarkazm ma natywne anotacje tylko w TwitterEmo. Dwa modele: LogReg + TF-IDF char (OvR na 9 etykietach) oraz HerBERT-base fine-tune na 9 etykietach (recepta z 05).

Porównanie: macro-F1 **po 8 emocjach** modelu 9-etykietowego vs kanon 8-etykietowy. Progi na val, test nietknięty.

In [1]:
import pickle, warnings
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score
warnings.filterwarnings("ignore")

EMOTIONS=["radość","smutek","zaufanie","wstręt","strach","gniew","przeczuwanie","zdziwienie"]
LABELS9=EMOTIONS+["sarkazm"]
RANDOM_STATE=42
P=Path("../data/processed"); RESULTS_DIR=Path("../data/results"); CACHE=Path("../data/features")

tr=pd.read_csv(P/"twitteremo_train.csv"); va=pd.read_csv(P/"twitteremo_val.csv"); te=pd.read_csv(P/"twitteremo_test.csv")
for d in (tr,va,te): d["tekst"]=d["tekst"].fillna("")
print("sarkazm: train", int(tr.sarkazm.sum()), "val", int(va.sarkazm.sum()), "test", int(te.sarkazm.sum()))

def opt_thr(yt,yp):
    thr=np.full(yt.shape[1],0.5)
    for i in range(yt.shape[1]):
        bf,bt=0.0,0.5
        for t in np.arange(0.05,0.95,0.01):
            f=f1_score(yt[:,i],(yp[:,i]>=t).astype(int),zero_division=0)
            if f>bf: bf,bt=f,t
        thr[i]=bt
    return thr

sarkazm: train 584 val 123 test 44


## Część A: klasyczny (LogReg + TF-IDF char) — 8 vs 9 etykiet

In [2]:
TW=pickle.load(open(CACHE/"TW_FEATURES.pkl","rb"))["tfidf_char"]

def run_classical(labels):
    y_tr,y_va,y_te=tr[labels].values,va[labels].values,te[labels].values
    clf=OneVsRestClassifier(LogisticRegression(max_iter=1000,C=1.0,class_weight="balanced",
            solver="liblinear",random_state=RANDOM_STATE))
    clf.fit(TW["train"],y_tr)
    thr=opt_thr(y_va,clf.predict_proba(TW["val"]))
    pred=(clf.predict_proba(TW["test"])>=thr).astype(int)
    return y_te,pred

y8,p8=run_classical(EMOTIONS)
y9,p9=run_classical(LABELS9)

rows=[]
rows.append({"model":"LogReg 8-label (kanon)","f1_macro_8emo":f1_score(y8,p8,average="macro",zero_division=0),"f1_sarkazm":np.nan})
rows.append({"model":"LogReg 9-label","f1_macro_8emo":f1_score(y9[:,:8],p9[:,:8],average="macro",zero_division=0),
             "f1_sarkazm":f1_score(y9[:,8],p9[:,8],zero_division=0)})
clas=pd.DataFrame(rows)
print(clas.round(3).to_string(index=False))
print("\nsarkazm (LogReg 9-label): P=%.3f R=%.3f n_pos_test=%d"%(
    precision_score(y9[:,8],p9[:,8],zero_division=0), recall_score(y9[:,8],p9[:,8],zero_division=0), int(y9[:,8].sum())))

                 model  f1_macro_8emo  f1_sarkazm
LogReg 8-label (kanon)          0.474         NaN
        LogReg 9-label          0.474       0.192

sarkazm (LogReg 9-label): P=0.241 R=0.159 n_pos_test=44


## Część B: HerBERT-base — fine-tune na 9 etykietach

In [3]:
import torch, torch.nn.functional as F
from scipy.special import expit
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding, EarlyStoppingCallback)
torch.manual_seed(RANDOM_STATE); np.random.seed(RANDOM_STATE)
MODEL="allegro/herbert-base-cased"; device="cuda" if torch.cuda.is_available() else "cpu"
HF_OUT=Path("../data/transformers"); MAX_LEN=128
print("device:",device)

tok=AutoTokenizer.from_pretrained(MODEL)
def to_ds(df,labels):
    d=Dataset.from_dict({"text":df["tekst"].tolist(),"labels":df[labels].values.astype("float32").tolist()})
    return d.map(lambda b: tok(b["text"],truncation=True,max_length=MAX_LEN),batched=True,remove_columns=["text"])

y_tr9=tr[LABELS9].values
pos=y_tr9.sum(0); neg=len(y_tr9)-pos
POS_WEIGHT=torch.tensor(np.clip(neg/np.maximum(pos,1),1.0,10.0),dtype=torch.float32)
print("pos_weight:",dict(zip(LABELS9,POS_WEIGHT.numpy().round(2).tolist())))

class WT(Trainer):
    def __init__(self,*a,pos_weight=None,**k): super().__init__(*a,**k); self.pw=pos_weight
    def compute_loss(self,model,inputs,return_outputs=False,**kw):
        labels=inputs.pop("labels"); out=model(**inputs)
        loss=F.binary_cross_entropy_with_logits(out.logits.float(),labels.float(),
            pos_weight=self.pw.to(out.logits.device) if self.pw is not None else None)
        return (loss,out) if return_outputs else loss

ds_tr,ds_va,ds_te=to_ds(tr,LABELS9),to_ds(va,LABELS9),to_ds(te,LABELS9)
model=AutoModelForSequenceClassification.from_pretrained(MODEL,num_labels=9,problem_type="multi_label_classification")
args=TrainingArguments(output_dir=str(HF_OUT/"sarcasm_9label"),eval_strategy="epoch",save_strategy="epoch",
    save_total_limit=1,load_best_model_at_end=True,metric_for_best_model="f1_macro",greater_is_better=True,
    per_device_train_batch_size=8,per_device_eval_batch_size=32,gradient_accumulation_steps=2,
    gradient_checkpointing=True,num_train_epochs=8,learning_rate=2e-5,warmup_ratio=0.1,weight_decay=0.01,
    fp16=(device=="cuda"),logging_steps=200,report_to="none",seed=RANDOM_STATE)
cm=lambda p:{"f1_macro":f1_score(p.label_ids.astype(int),(expit(p.predictions)>=0.5).astype(int),average="macro",zero_division=0)}
trainer=WT(model=model,args=args,train_dataset=ds_tr,eval_dataset=ds_va,data_collator=DataCollatorWithPadding(tok),
    compute_metrics=cm,pos_weight=POS_WEIGHT,callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
trainer.train()

device: cuda


pos_weight: {'radość': 7.71999979019165, 'smutek': 10.0, 'zaufanie': 10.0, 'wstręt': 3.309999942779541, 'strach': 10.0, 'gniew': 4.659999847412109, 'przeczuwanie': 1.850000023841858, 'zdziwienie': 10.0, 'sarkazm': 10.0}


Map:   0%|          | 0/28684 [00:00<?, ? examples/s]

Map:  14%|█▍        | 4000/28684 [00:00<00:00, 34338.24 examples/s]

Map:  28%|██▊       | 8000/28684 [00:00<00:00, 34935.58 examples/s]

...

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.16it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.16it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

There were unexpected keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.beta', 'bert.embeddings.LayerNorm.gamma', 'bert.encoder.layer.0.attention.output.LayerNorm.beta', 'bert.encoder.layer.0.attention.output.LayerNorm.gamma', 'bert.encoder.layer.0.output.LayerNorm.beta', 'bert.encoder.layer.0.output.LayerNorm.gamma', 'bert.encoder.layer.1.attention.output.LayerNorm.beta', 'bert.encoder.layer.1.attention.output.LayerNorm.gamma', 'bert.encoder.layer.1.output.LayerNorm.beta', 'bert.encoder.layer.1.output.LayerNorm.gamma', 'bert.encoder.layer.2.attention.output.LayerNorm.beta', 'bert.encoder.layer.2.attention.output.LayerNorm.gamma', 'bert.encoder.layer.2.output.LayerNorm.beta', 'bert.encoder.layer.2.output.LayerNorm.gamma', 'bert.encoder.layer.3.attention.output.LayerNorm.beta', 'bert.encoder.layer.3.attention.output.LayerNorm.gamma', 'bert.encoder.layer.3.output.LayerNorm.beta', 'bert.encoder.layer.3.output.LayerNorm.gamma', 'bert.encoder.layer.4.attention.output.LayerNor

TrainOutput(global_step=12551, training_loss=0.7579491938759004, metrics={'train_runtime': 2068.461, 'train_samples_per_second': 110.939, 'train_steps_per_second': 6.935, 'total_flos': 8058467514628080.0, 'train_loss': 0.7579491938759004, 'epoch': 7.0})

In [4]:
pv=expit(trainer.predict(ds_va).predictions); pt=expit(trainer.predict(ds_te).predictions)
y_va9,y_te9=va[LABELS9].values,te[LABELS9].values
thr=opt_thr(y_va9,pv); pred=(pt>=thr).astype(int)

f1_8=f1_score(y_te9[:,:8],pred[:,:8],average="macro",zero_division=0)
f1_s=f1_score(y_te9[:,8],pred[:,8],zero_division=0)
ref=pd.read_csv(RESULTS_DIR/"transformers_results.csv")
ref8=float(ref.loc[ref["model"].str.contains("herbert-base",case=False),"f1_macro"].iloc[0])

res=pd.DataFrame([
 {"model":"HerBERT-base 8-label (kanon, 05)","f1_macro_8emo":ref8,"f1_sarkazm":np.nan},
 {"model":"HerBERT-base 9-label","f1_macro_8emo":f1_8,"f1_sarkazm":f1_s},
])
out=pd.concat([clas,res],ignore_index=True)
out.round(3).to_csv(RESULTS_DIR/"sarcasm_9label.csv",index=False)
display(out.round(3))
print("sarkazm (HerBERT 9-label): P=%.3f R=%.3f"%(
    precision_score(y_te9[:,8],pred[:,8],zero_division=0), recall_score(y_te9[:,8],pred[:,8],zero_division=0)))

d_clf=clas.loc[1,"f1_macro_8emo"]-clas.loc[0,"f1_macro_8emo"]


,model,f1_macro_8emo,f1_sarkazm
0,LogReg 8-label (kanon),0.474,NaN
1,LogReg 9-label,0.474,0.192
2,"HerBERT-base 8-label (kanon, 05)",0.551,NaN
3,HerBERT-base 9-label,0.552,0.272


sarkazm (HerBERT 9-label): P=0.297 R=0.250
# Sarkazm jako 9. etykieta — wnioski

- PB-1 (wykrywalność): F1 sarkazmu — LogReg 0.192, HerBERT 0.272
  (n_pos: train 584, test 44).
- PB-2 (interferencja): macro-F1 po 8 emocjach — LogReg 9-label 0.474
  vs kanon 0.474 (delta +0.000); HerBERT 9-label 0.552 vs kanon 0.551 (delta +0.001).
- Kontekst: sarkastyczne tweety maja +45% bledow etykiet (sarcasm_analysis.md).

